# Week 5: Model Interpretability

## Learning Objectives
- Compute permutation importance
- Create partial dependence plots
- Use SHAP values for explanations
- Explain individual predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')

## 1. Permutation Importance

In [ ]:
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

result = permutation_importance(rf, X_test, y_test, n_repeats=20,
                                  random_state=42, scoring='roc_auc')
imp_df = pd.DataFrame({'feature': X.columns,
                        'importance': result.importances_mean,
                        'std': result.importances_std}
                      ).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(8, 5))
sns.barplot(data=imp_df, x='importance', y='feature')
plt.title('Permutation Importance — Top 10')
plt.tight_layout(); plt.show()

## 2. Partial Dependence Plots

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

top2_idx = np.argsort(gb.feature_importances_)[::-1][:2]
fig, ax = plt.subplots(1, 1, figsize=(10, 3))
PartialDependenceDisplay.from_estimator(
    gb, X_train, features=list(top2_idx),
    feature_names=list(X.columns), ax=ax)
plt.suptitle('Partial Dependence Plots', y=1.02)
plt.tight_layout(); plt.show()

## 3. SHAP Values

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_test)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    shap.summary_plot(sv, X_test, feature_names=list(X.columns), show=False)
    plt.title('SHAP Summary Plot')
    plt.tight_layout(); plt.show()
except ImportError:
    print('SHAP not installed: pip install shap')

## Exercise

1. Compare tree feature_importances_ with permutation importance
2. Create PDP for the top 4 features in a GradientBoosting model
3. Use SHAP to explain a single misclassified sample

In [ ]:
# Your code here


## Summary

- ✓ Permutation feature importance
- ✓ Partial Dependence Plots
- ✓ SHAP for global and local explanation

## Next Month
Deep Learning!